# Coeficientes de Floquet

Vamos a armar un programa que pueda hacer una tabla de inestabilidades para cualquier tipo de potencial $V(\phi)$.

## Física

**$\LARGE{\text{Ecuaciones dinámicas}}$**

Durante recalentamiento tenemos que las ecuaciones de movimiento que rigen el background y las perturbaciones son las siguientes:

$$\ddot{\phi} + 3 \frac{\dot{a}}{a} \dot{\phi} + \frac{dV}{d\phi} = 0$$

$$\delta \ddot{\phi}_k + 3 \frac{\dot{a}}{a} ~ \delta \dot{\phi}_k + \left[ \frac{k^2}{a^2} + \frac{d^2 V}{d\phi^2} \right] \delta \phi_k = 0$$

$$\ddot{\chi}_k + 3 \frac{\dot{a}}{a} \dot{\chi}_k + \left( \frac{k^2}{a^2} + g^2 \phi^2 \right) \chi_k = 0$$

$$\ddot{a} + \frac{a}{3 m_p^2} \left[ \dot{\phi}^2 - V (\phi) \right] = 0$$

Y para hacer un análisis de tipo Floquet necesitamos que el universo no se expanda lo que implica tomar $a = 1 = cte$ y $H = 0$, por tanto tenemos que en ese caso las ecuaciones nos quedan de la siguiente forma:

$$\ddot{\phi} + \frac{dV}{d\phi}= 0$$

$$\delta \ddot{\phi}_k + \left( k^2 + \frac{d^2 V}{d\phi^2} \right) \delta \phi_k = 0$$

$$\ddot{\chi}_k + \left( k^2 + g^2 \phi^2 \right) \chi_k = 0$$

**$\LARGE{\text{Conservación de la energía}}$**

Para hacer un análisis de tipo Floquet necesitamos conocer el período de oscilación del inflatón y para ello usaremos la conservación de la energía, la cual surge de plantear que el universo no se expande:

- A tiempo inicial para reheating tenemos 

$$E = \frac{\dot{\phi}_0^2}{2} + V(\phi_0)$$

- Y para un tiempo arbitrario tenemos 

$$E = \frac{\dot{\phi}^2}{2} + V(\phi)$$

**$\LARGE{\text{Período de oscilación del inflatón}}$**

Despejando de la última ecuación tenemos

$$\dot{\phi} = \sqrt{2 \left[E - V \left( \phi \right) \right]}$$

$$\frac{d\phi}{dt} = \sqrt{2 \left[E - V \left( \phi \right) \right]}$$

$$dt = \frac{d\phi}{\sqrt{2 \left[E - V \left( \phi \right) \right]}}$$

$$\int_0^T dt = 2 \int_{-\phi_{max}}^{\phi_{max}} \frac{d\phi}{\sqrt{2 \left[E - V \left( \phi \right) \right]}}$$

$$T = \frac{4}{\sqrt{2 E}} \int_{0}^{\phi_{max}} \frac{d\phi}{\displaystyle \sqrt{1- \frac{V \left( \phi \right)}{E}}}$$

**$\LARGE{m^2 \phi^2}$**

Veamos cuánto vale para $V(\phi) = m^2 \phi^2$ para verificar que da una constante

$$T = \frac{4}{\sqrt{4 E}} \int_{0}^{\phi_{max}} \frac{d\phi}{\displaystyle \sqrt{1- \frac{m^2 \phi^2}{2E}}}$$

- Definimos $\displaystyle \sin (u) = \frac{m \phi}{\sqrt{2E}}$ y $\displaystyle \cos (u) ~ du = \frac{m}{\sqrt{2E}} ~ d\phi$, tal que la integral nos queda de la siguiente forma

$$T = \frac{2}{m} \int^{-\arcsin \left( \frac{m \phi_{max}}{\sqrt{2E}} \right)}_{0} \frac{\cos u ~ du}{\displaystyle \sqrt{1- \sin^2 u}}$$

$$T = \frac{2}{m} \int^{-\arcsin \left( \frac{m \phi_{max}}{\sqrt{2E}} \right)}_0 \frac{\cos u ~ du}{\displaystyle \cos u}$$

$$T = \frac{2}{m} \int^{-\arcsin \left( \frac{m \phi_{max}}{\sqrt{2E}} \right)}_{0} du$$

$$T = \frac{2}{m} \arcsin \left( \frac{m \phi_{max}}{\sqrt{2E}} \right)$$

$$T = \frac{2}{m} \arcsin \left( \frac{m \phi_{max}}{\sqrt{2E}} \right)$$

Podemos ver que el valor del máximo del campo sale de $E = V(\phi_{max})$ que en el caso de $\displaystyle V(\phi) = \frac{1}{2} m^2 \phi^2$ tenemos $\displaystyle \phi_{max} = \frac{\sqrt{2E}}{m}$ y por tanto

$$T = \frac{2}{m} \arcsin \left( \frac{m \sqrt{2E}}{m \sqrt{2E}} \right)$$

$$T = \frac{2\pi}{m}$$

## Librerías

In [1]:
import os
import numpy as np
from tqdm import tqdm 
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.signal import find_peaks
from scipy.optimize import curve_fit
from scipy.integrate import solve_ivp

mpl.rc('figure', figsize=(12, 6))
mpl.rc('text', usetex = False)
mpl.rc('font', family = 'serif')
mpl.rc('font', size = '14')
mpl.rc('xtick', labelsize=14) 
mpl.rc('ytick', labelsize=14)
mpl.rcParams['mathtext.fontset'] = 'stix'

In [2]:
# Función para colocar texto automáticamente en un lugar que no tape la curva
def place_text_auto(text, ax=None, x_rel=0.8, y_rel=0.9, **kwargs):
    """
    Coloca texto en el gráfico en una posición relativa al eje,
    para evitar tapar la curva.
    
    Parámetros:
    - text: texto a colocar
    - ax: eje de matplotlib (opcional, usa gca() por defecto)
    - x_rel: posición relativa en x (0 a 1, 0.8 = 80% del ancho)
    - y_rel: posición relativa en y (0 a 1, 0.9 = 90% del alto)
    - **kwargs: argumentos adicionales para plt.text
    """
    if ax is None:
        ax = plt.gca()
    
    # Obtener límites del eje
    x_min, x_max = ax.get_xlim()
    y_min, y_max = ax.get_ylim()
    
    # Calcular posición absoluta
    x_pos = x_min + x_rel * (x_max - x_min)
    y_pos = y_min + y_rel * (y_max - y_min)
    
    ax.text(x_pos, y_pos, text, **kwargs)

In [ ]:
# Armamos la función para guardar los datos en un archivo CSV

def save_csv(qs, Aks, mu, filename='results'):
    output = np.zeros((len(Aks)+1, len(qs)+1))
    output[0, 1:] = qs
    output[1:, 0] = Aks
    output[1:, 1:] = np.real(mu)
    np.savetxt(f"{filename}.csv", output, delimiter=",")


def final_state(sol):
    """Extrae el estado final del integrador solve_ivp de forma robusta."""
    y = sol.y
    if isinstance(y, list):
        y = np.asarray(y)
    y = np.asarray(y)
    if y.ndim == 1:
        return y
    return y[:, -1]

## $m^2 \phi^2$

Correcciones a tener en cuenta:
- Cuidado que g y q son distintas, se relacionan a partir de $\displaystyle q = \frac{g^2 \tilde{\phi}_0^2}{4}$.
- $\displaystyle \phi_0 = \frac{m_p}{m} \simeq 10^6$.
- Hay que tener en cuenta que $\displaystyle A_k = \tilde{k}^2 + \frac{g^2 \tilde{\phi}_0^2}{2}$. 

### Funciones

In [ ]:
# Función para armarse la matriz de monodromía y calcular los coeficientes de Floquet
def mu_m2phi2(k, g, y0, Tosc):
    phi0, phi_dot0 = y0

    def pert(t, y, k, g):
        phi, phi_dot, chi_k, chi_dot_k = y
        phi_ddot = - phi
        chi_ddot_k = - (k**2 + g**2 * phi**2) * chi_k
        return [phi_dot, phi_ddot, chi_dot_k, chi_ddot_k]

    t_span = (0, Tosc)

    sol1 = solve_ivp(pert, t_span, [phi0, phi_dot0, 1.0, 0.0], t_eval=[Tosc], method='RK45', rtol=1e-6, args=(k, g))
    y_final1 = final_state(sol1)

    sol2 = solve_ivp(pert, t_span, [phi0, phi_dot0, 0.0, 1.0], t_eval=[Tosc], method='RK45', rtol=1e-6, args=(k, g))
    y_final2 = final_state(sol2)

    M = np.array([
        [y_final1[2], y_final2[2]],
        [y_final1[3], y_final2[3]]
    ])

    tr_M = np.trace(M)
    argument = np.abs(tr_M / 2.0)

    if argument > 1.0:
        mu = np.arccosh(argument) / Tosc
        return mu
    else:
        return 0.0
    
# Función para calcular el período de oscilación del inflatón
def oscilation_m2phi2(E):
    def integrand(phi):
        V = 0.5 * phi**2
        return 1.0 / np.sqrt(2 * (E - V))
    phimax = np.sqrt(2*E)
    integral, _ = quad(integrand, 0, phimax, args=())
    Tosc = 4*integral
    return Tosc

In [ ]:
# Verifico si el archivo de inestabilidades existe, sino lo armo
filename = "mathieu_m2phi2.csv"

if not os.path.exists(filename):
    phi0 = 1e6
    phi_dot0 = 0
    y0 = (phi0, phi_dot0)
    E = 0.5 * phi_dot0**2 + 0.5 * phi0**2

    qs = np.linspace(0, 5, 300)
    ks = np.logspace(-2, 1, 300)
    gs = []

    for q in qs:
        g = 2 * np.sqrt(q) / phi0
        gs.append(g)

    G, K = np.meshgrid(gs, ks)
    mu = np.zeros((len(ks), len(gs)), dtype=complex)
    Tosc = oscilation_m2phi2(E)

    for i, g in enumerate(tqdm(gs)):
        for j, k in enumerate(ks):
            mu[j, i] = mu_m2phi2(k, g, y0, Tosc)

    # Guardemos los resultados en un archivo CSV
    save_csv(gs, ks, mu, filename='mathieu_m2phi2')
    
else:
    data = np.loadtxt(filename, delimiter=",")
    gs = data[0, 1:]
    ks = data[1:, 0]
    mu = data[1:, 1:]

In [ ]:
# Ahora grafiquemos el diagrama de inestabilidades
plt.figure(figsize=(10, 6))
plt.contourf(gs, ks, np.real(mu), levels=100, cmap='inferno')
plt.colorbar(label=r'$\mu_k$')
plt.xlabel(r'$g$')
plt.ylabel(r'$k$')
plt.title(r'Diagrama de inestabilidades de Floquet para $m^2 \phi^2$')
plt.yscale('log')
plt.tight_layout()
plt.savefig("Floquet_m2phi2.pdf", format = "pdf")

In [ ]:
# Diagrama del coeficientente de Floquet en base a k para un g fijo
filename = "mathieu_m2phi2.csv"

data = np.loadtxt(filename, delimiter=",")
gs = data[0, 1:]
ks = data[1:, 0]
mu = data[1:, 1:]

G = np.linspace(1e-6, 2e-6, 16)
i = 0

plt.figure(figsize=(20, 20))
plt.xlabel(r'$k$', fontsize=30, labelpad=40)
plt.ylabel(r'$\mu_k$', fontsize=30, labelpad=75)
plt.xticks([])
plt.yticks([])

for g in G:
    i += 1
    index = (np.abs(gs - g)).argmin()

    plt.subplot(4, 4, i)
    plt.plot(ks, np.real(mu[:, index]), label=f'$g={g}$', lw = 5, color = "#38771e")# "#88086c")
    ymin, ymax = plt.gca().get_ylim()
    if i == 3:
        plt.text(x=ks[10], y=ymax*0.8, s=rf'$g = {gs[index]:.2e}$', fontsize=18, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))
    elif i == 13 or i == 14:
        plt.text(x=ks[-130], y=ymax*0.8, s=rf'$g = {gs[index]:.2e}$', fontsize=18, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))
    else:
        plt.text(x=ks[10], y=ymax*0.2, s=rf'$g = {gs[index]:.2e}$', fontsize=18, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))
    plt.xscale('log')
    plt.grid(alpha = 0.3)

plt.tight_layout()
#plt.show()
plt.savefig("muk_m2phi2.pdf", format = "pdf")

In [ ]:
# Diagrama del coeficientente de Floquet en base a k para un g fijo
filename = "mathieu_m2phi2.csv"


data = np.loadtxt(filename, delimiter=",")
gs = data[0, 1:]
ks = data[1:, 0]
mu = data[1:, 1:]


G = np.linspace(1e-6, 3e-6, 6)
i = 0


plt.figure(figsize=(15, 8))
plt.xlabel(r'$k$', fontsize=30, labelpad=40)
plt.ylabel(r'$\mu_k$', fontsize=30, labelpad=60, rotation=0)
plt.xticks([])
plt.yticks([])


for g in G:
    i += 1
    index = (np.abs(gs - g)).argmin()

    max_mu = np.max(np.real(mu[:, index]))
    k_max = ks[np.argmax(np.real(mu[:, index]))]
    k_delta = ks[np.where(np.real(mu[:, index]) >= max_mu/2)[0]]

    ax = plt.subplot(2, 3, i)
    plt.plot(ks, np.real(mu[:, index]), label=f'$g={g}$', lw = 5, color = "#38771e")# "#88086c")
    ymin, ymax = plt.gca().get_ylim()
    # Formatear g para mostrar \times 10^6 en vez de e-06
    g_fmt = f"{gs[index]*1e6:.2f}"
    # Ajustar la posición del texto para el primer subplot
    if i == 1:
        # Colocar el texto más adentro y ajustar límites para evitar solapamiento con el eje
        plt.xlim(left=ks[0]*0.6)
        plt.text(x=ks[10], y=ymax*0.7, s=rf'$g = {g_fmt} \times 10^{{-6}}$', fontsize=18, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))
    elif i == 2:
        plt.yticks(np.arange(0,0.25,0.04))
        plt.text(x=ks[5], y=ymax*0.2, s=rf'$g = {g_fmt} \times 10^{{-6}}$', fontsize=18, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))
    elif i == 5:
        plt.yticks(np.arange(0,0.20,0.03))
        plt.text(x=ks[5], y=ymax*0.2, s=rf'$g = {g_fmt} \times 10^{{-6}}$', fontsize=18, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))
    elif i == 3:
        plt.text(x=ks[-140], y=ymax*0.8, s=rf'$g = {g_fmt} \times 10^{{-6}}$', fontsize=18, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))
    else:
        plt.text(x=ks[5], y=ymax*0.2, s=rf'$g = {g_fmt} \times 10^{{-6}}$', fontsize=18, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))
    plt.xscale('log')
    plt.grid(alpha = 0.3)
    plt.axvspan(k_delta[-1], k_delta[0], color='orange', alpha=0.5)


plt.tight_layout()
#plt.show()
plt.savefig("muk_m2phi2.pdf", format = "pdf")

In [ ]:
phi0 = 1e6
phi_dot0 = 0#phi0

y0 = (phi0, phi_dot0)
E = 0.5 * phi_dot0**2 + 0.5 * phi0**2

q = 1e-6
ks = np.logspace(-2, 2, 1000)

# Q, K = np.meshgrid(qs, ks)
mu = np.zeros(len(ks), dtype=complex)
Tosc = oscilation_m2phi2(E)

for j, k in enumerate(tqdm(ks)):
    mu[j] = mu_m2phi2(k, q, y0, Tosc)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(ks, np.real(mu), label=f'$q={q}$', lw = 5, color = "#88086c")
plt.xlabel(r'$k$')
plt.ylabel(r'$\mu_k$')
plt.xscale('log')
plt.grid(alpha = 0.3)
plt.text(x=ks[10], y=0.7*max(np.real(mu)), s=rf'$g = {q}$', fontsize=18, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))
plt.text(x=ks[10], y=0.6*max(np.real(mu)), s=rf'$k_m = {ks[np.argmax(np.real(mu))]}$', fontsize=18, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))
plt.tight_layout()
plt.show()
print(ks[np.argmax(np.real(mu))])

In [ ]:
phi0 = 1e6
phi_dot0 = 0#phi0

y0 = (phi0, phi_dot0)
E = 0.5 * phi_dot0**2 + 0.5 * phi0**2

q = 4e-5
ks = np.logspace(-2, 2, 1000)

# Q, K = np.meshgrid(qs, ks)
mu = np.zeros(len(ks), dtype=complex)
Tosc = oscilation_m2phi2(E)

for j, k in enumerate(tqdm(ks)):
    mu[j] = mu_m2phi2(k, q, y0, Tosc)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(ks, np.real(mu), label=f'$q={q}$', lw = 5, color = "#88086c")
plt.xlabel(r'$k$')
plt.ylabel(r'$\mu_k$')
plt.xscale('log')
plt.grid(alpha = 0.3)
plt.text(x=ks[10], y=0.7*max(np.real(mu)), s=rf'$g = {q}$', fontsize=18, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))
plt.text(x=ks[10], y=0.6*max(np.real(mu)), s=rf'$k_m = {ks[np.argmax(np.real(mu))]}$', fontsize=18, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))
plt.tight_layout()
plt.show()
print(ks[np.argmax(np.real(mu))])

## $\lambda \phi^4$

Consideraciones:
- La ecuación adimensional es 

$$\ddot{\tilde{\chi}}_k + \left( \tilde{k}^2 + q \tilde{\phi}^2 \right) \tilde{\chi}_k = 0$$

dónde $\displaystyle q = \frac{g^2}{\lambda}$.

In [ ]:
# Función para armarse la matriz de monodromía y calcular los coeficientes de Floquet
def mu_lphi4(k, q, y0, Tosc):
    phi0, phi_dot0 = y0

    def pert(t, y, k, q):
        phi, phi_dot, chi_k, chi_dot_k = y
        phi_ddot = - phi**3
        chi_ddot_k = - (k**2 + q * phi**2) * chi_k
        return [phi_dot, phi_ddot, chi_dot_k, chi_ddot_k]

    t_span = (0, Tosc)

    sol1 = solve_ivp(pert, t_span, [phi0, phi_dot0, 1.0, 0.0], t_eval=[Tosc], method='RK45', rtol=1e-6, args=(k, q))
    y_final1 = final_state(sol1)

    sol2 = solve_ivp(pert, t_span, [phi0, phi_dot0, 0.0, 1.0], t_eval=[Tosc], method='RK45', rtol=1e-6, args=(k, q))
    y_final2 = final_state(sol2)

    M = np.array([
        [y_final1[2], y_final2[2]],
        [y_final1[3], y_final2[3]]
    ])

    tr_M = np.trace(M)
    argument = np.abs(tr_M / 2.0)

    if argument > 1.0:
        mu = np.arccosh(argument) / Tosc
        return mu
    else:
        return 0.0
    
# Función para calcular el período de oscilación del inflatón
def oscilation_lphi4(E):
    def integrand(phi):
        V = 0.25 * phi**4
        return 1.0 / np.sqrt(2 * (E - V))
    phimax = (4*E)**(1/4)
    integral, _ = quad(integrand, 0, phimax)
    Tosc = 4*integral
    return Tosc

In [ ]:
# Verifico si el archivo de inestabilidades existe, sino lo armo
filename = "mathieu_lphi4(2).csv"

if not os.path.exists(filename):
    phi0 = 1
    phi_dot0 = phi0**2/np.sqrt(2)

    y0 = (phi0, phi_dot0)
    E = 0.5 * phi_dot0**2 + 0.25 * phi0**4

    qs = np.linspace(0, 25, 300)
    ks = np.linspace(0, 2, 300)

    # Q, K = np.meshgrid(qs, ks)
    mu = np.zeros((len(ks), len(qs)), dtype=complex)
    Tosc = oscilation_lphi4(E)

    for i, q in enumerate(tqdm(qs)):
        for j, k in enumerate(ks):
            mu[j, i] = mu_lphi4(k, q, y0, Tosc)

    # Guardemos los resultados en un archivo CSV
    save_csv(qs, ks, mu, filename="mathieu_lphi4(2)")
    
else:
    data = np.loadtxt(filename, delimiter=",")
    qs = data[0, 1:]
    ks = data[1:, 0]
    mu = data[1:, 1:]

In [ ]:
# Ahora grafiquemos el diagrama de inestabilidades
plt.figure(figsize=(10, 6))
plt.contourf(qs, ks, np.real(mu), levels=100, cmap='inferno')
cbar = plt.colorbar(orientation='vertical')
cbar.set_label(r'$\Re(\mu_k)$', fontsize=20, labelpad=25, rotation=0)
plt.xlabel(r'$q$', fontsize=20)
plt.ylabel(r'$k$', fontsize=20, labelpad=20, rotation=0)
#plt.title(r'Diagrama de inestabilidades de Floquet para $\lambda \phi^4$')
plt.tight_layout()
plt.savefig("Floquet_lphi4.pdf", format = "pdf")

In [ ]:
# Diagrama del coeficientente de Floquet en base a k para un g fijo
filename = "mathieu_lphi4(2).csv"

data = np.loadtxt(filename, delimiter=",")
qs = data[0, 1:]
ks = data[1:, 0]
mu = data[1:, 1:]

Q = np.linspace(0.5, 3.5, 16)
i = 0

plt.figure(figsize=(20, 20))
plt.xlabel(r'$k$', fontsize=30, labelpad=40)
plt.ylabel(r'$\mu_k$', fontsize=30, labelpad=75)
plt.xticks([])
plt.yticks([])

for q in Q:
    i += 1
    index = (np.abs(qs - q)).argmin()

    plt.subplot(4, 4, i)
    plt.plot(ks, np.real(mu[:, index]), label=f'$q={q}$', lw = 5, color = "#38771e")# "#88086c")
    ymin, ymax = plt.gca().get_ylim()
    if i == 14 or i == 15 or i == 16:
        plt.text(x=ks[10], y=ymax*0.8, s=rf'$q = {qs[index]:.2f}$', fontsize=18, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))
    else:
        plt.text(x=ks[-80], y=ymax*0.8, s=rf'$q = {qs[index]:.2f}$', fontsize=18, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))
    plt.grid(alpha = 0.3)

plt.tight_layout()
#plt.show()
plt.savefig("muk_lphi4.pdf", format = "pdf")

In [ ]:
# Diagrama del coeficientente de Floquet en base a k para un g fijo
filename = "mathieu_lphi4(2).csv"

data = np.loadtxt(filename, delimiter=",")
qs = data[0, 1:]
ks = data[1:, 0]
mu = data[1:, 1:]

Q = np.linspace(0.5, 3.5, 6)
i = 0

plt.figure(figsize=(15, 8))
plt.xlabel(r'$k$', fontsize=30, labelpad=40)
plt.ylabel(r'$\mu_k$', fontsize=30, labelpad=60, rotation=0)
plt.xticks([])
plt.yticks([])

for q in Q:
    i += 1
    index = (np.abs(qs - q)).argmin()

    max_mu = np.max(np.real(mu[:, index]))
    k_max = ks[np.argmax(np.real(mu[:, index]))]
    k_delta = ks[np.where(np.real(mu[:, index]) >= max_mu/2)[0]]

    plt.subplot(2, 3, i)
    plt.plot(ks, np.real(mu[:, index]), label=f'$q={q}$', lw = 5, color = "#38771e")# "#88086c")
    ymin, ymax = plt.gca().get_ylim()
    if i == 6:
        plt.text(x=ks[10], y=ymax*0.8, s=rf'$q = {qs[index]:.2f}$', fontsize=18, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))
    else:
        plt.text(x=ks[-80], y=ymax*0.8, s=rf'$q = {qs[index]:.2f}$', fontsize=18, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))
    plt.grid(alpha = 0.3)
    plt.axvspan(k_delta[-1], k_delta[0], color='orange', alpha=0.5)

plt.tight_layout()
#plt.show()
plt.savefig("muk_lphi4.pdf", format = "pdf")

In [ ]:
phi0 = 1
phi_dot0 = 0#phi0**2/np.sqrt(2)

y0 = (phi0, phi_dot0)
E = 0.5 * phi_dot0**2 + 0.25 * phi0**4

q = 5050
ks = np.logspace(-2, 2, 1000)

# Q, K = np.meshgrid(qs, ks)
mu = np.zeros(len(ks), dtype=complex)
Tosc = oscilation_lphi4(E)

for j, k in enumerate(tqdm(ks)):
    mu[j] = mu_lphi4(k, q, y0, Tosc)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(ks, np.real(mu), label=f'$q={q}$', lw = 5, color = "#88086c")
plt.xlabel(r'$k$')
plt.ylabel(r'$\mu_k$')
plt.xscale('log')
plt.grid(alpha = 0.3)
plt.text(x=ks[10], y=0.7*max(np.real(mu)), s=rf'$q = {q}$', fontsize=18, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))
plt.text(x=ks[10], y=0.6*max(np.real(mu)), s=rf'$k_m = {ks[np.argmax(np.real(mu))]}$', fontsize=18, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))
plt.tight_layout()
plt.show()

In [ ]:
phi0 = 1
phi_dot0 = 0#phi0**2/np.sqrt(2)

y0 = (phi0, phi_dot0)
E = 0.5 * phi_dot0**2 + 0.25 * phi0**4

q = 0.5
ks = np.logspace(-2, 2, 1000)

# Q, K = np.meshgrid(qs, ks)
mu = np.zeros(len(ks), dtype=complex)
Tosc = oscilation_lphi4(E)

for j, k in enumerate(tqdm(ks)):
    mu[j] = mu_lphi4(k, q, y0, Tosc)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(ks, np.real(mu), label=f'$q={q}$', lw = 5, color = "#88086c")
plt.xlabel(r'$k$')
plt.ylabel(r'$\mu_k$')
plt.xscale('log')
plt.grid(alpha = 0.3)
plt.text(x=ks[10], y=0.7*max(np.real(mu)), s=rf'$q = {q}$', fontsize=18, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))
plt.text(x=ks[10], y=0.6*max(np.real(mu)), s=rf'$k_m = {ks[np.argmax(np.real(mu))]}$', fontsize=18, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))
plt.tight_layout()
plt.show()

print(ks[np.argmax(np.real(mu))])

## Con expansión

In [6]:
# Función para armarse la matriz de monodromía y calcular los coeficientes de Floquet
def mu_lphi4(k, q, y0, Tosc):
    phi0, phi_dot0 = y0

    def pert(t, y, k, q):
        phi, phi_dot, chi_k, chi_dot_k, a, a_dot = y

        # Evolución de a(t)
        a_ddot = - a/3 * (phi_dot**2 - phi**4/4)
        H = a_dot/a
        
        # Ecuación para φ
        phi_ddot = -3 * H * phi_dot - phi **3
        
        # Ecuaciones para χ_k
        chi_ddot_k = - 3 * H * chi_dot_k - (k ** 2 / a**2 + q * phi**2) * chi_k 
        
        return [phi_dot, phi_ddot, chi_dot_k, chi_ddot_k, a_dot, a_ddot]

    t_span = (0, Tosc)

    sol1 = solve_ivp(pert, t_span, [phi0, phi_dot0, 1.0, 0.0, 1.0, 0.0], t_eval=[Tosc], method='RK45', rtol=1e-6, args=(k, q))
    y_final1 = final_state(sol1)

    sol2 = solve_ivp(pert, t_span, [phi0, phi_dot0, 0.0, 1.0, 1.0, 0.0], t_eval=[Tosc], method='RK45', rtol=1e-6, args=(k, q))
    y_final2 = final_state(sol2)

    M = np.array([
        [y_final1[2], y_final2[2]],
        [y_final1[3], y_final2[3]]
    ])

    tr_M = np.trace(M)
    argument = np.abs(tr_M / 2.0)

    if argument > 1.0:
        mu = np.arccosh(argument) / Tosc
        return mu
    else:
        return 0.0
    
# Función para calcular el período de oscilación del inflatón
def oscilation_lphi4(E):
    def integrand(phi):
        V = 0.25 * phi**4
        return 1.0 / np.sqrt(2 * (E - V))
    phimax = (4*E)**(1/4)
    integral, _ = quad(integrand, 0, phimax)
    Tosc = 4*integral
    return Tosc

In [7]:
# Verifico si el archivo de inestabilidades existe, sino lo armo
filename = "mathieu_lphi4(exp).csv"

if not os.path.exists(filename):
    phi0 = 1
    phi_dot0 = phi0**2/np.sqrt(2)

    y0 = (phi0, phi_dot0)
    E = 0.5 * phi_dot0**2 + 0.25 * phi0**4

    qs = np.linspace(0, 25, 300)
    ks = np.linspace(0, 2, 300)

    # Q, K = np.meshgrid(qs, ks)
    mu = np.zeros((len(ks), len(qs)), dtype=complex)
    Tosc = oscilation_lphi4(E)

    for i, q in enumerate(tqdm(qs)):
        for j, k in enumerate(ks):
            mu[j, i] = mu_lphi4(k, q, y0, Tosc)

    # Guardemos los resultados en un archivo CSV
    save_csv(qs, ks, mu, filename="mathieu_lphi4(exp)")
    
else:
    data = np.loadtxt(filename, delimiter=",")
    qs = data[0, 1:]
    ks = data[1:, 0]
    mu = data[1:, 1:]

  0%|          | 0/300 [00:00<?, ?it/s]


IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [ ]:
# Ahora grafiquemos el diagrama de inestabilidades
plt.figure(figsize=(10, 6))
plt.contourf(qs, ks, np.real(mu), levels=100, cmap='inferno')
cbar = plt.colorbar(orientation='vertical')
cbar.set_label(r'$\Re(\mu_k)$', fontsize=20, labelpad=25, rotation=0)
plt.xlabel(r'$q$', fontsize=20)
plt.ylabel(r'$k$', fontsize=20, labelpad=20, rotation=0)
#plt.title(r'Diagrama de inestabilidades de Floquet para $\lambda \phi^4$')
plt.tight_layout()
plt.savefig("Floquet_lphi4.pdf", format = "pdf")

In [ ]:
# Diagrama del coeficientente de Floquet en base a k para un g fijo
filename = "mathieu_lphi4(exp).csv"

data = np.loadtxt(filename, delimiter=",")
qs = data[0, 1:]
ks = data[1:, 0]
mu = data[1:, 1:]

Q = np.linspace(0.5, 3.5, 6)
i = 0

plt.figure(figsize=(15, 8))
plt.xlabel(r'$k$', fontsize=30, labelpad=40)
plt.ylabel(r'$\mu_k$', fontsize=30, labelpad=60, rotation=0)
plt.xticks([])
plt.yticks([])

for q in Q:
    i += 1
    index = (np.abs(qs - q)).argmin()

    max_mu = np.max(np.real(mu[:, index]))
    k_max = ks[np.argmax(np.real(mu[:, index]))]
    k_delta = ks[np.where(np.real(mu[:, index]) >= max_mu/2)[0]]

    plt.subplot(2, 3, i)
    plt.plot(ks, np.real(mu[:, index]), label=f'$q={q}$', lw = 5, color = "#38771e")# "#88086c")
    ymin, ymax = plt.gca().get_ylim()
    if i == 6:
        plt.text(x=ks[10], y=ymax*0.8, s=rf'$q = {qs[index]:.2f}$', fontsize=18, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))
    else:
        plt.text(x=ks[-80], y=ymax*0.8, s=rf'$q = {qs[index]:.2f}$', fontsize=18, bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.3'))
    plt.grid(alpha = 0.3)
    plt.axvspan(k_delta[-1], k_delta[0], color='orange', alpha=0.5)

plt.tight_layout()
#plt.show()
plt.savefig("muk_lphi4.pdf", format = "pdf")

# $\displaystyle \frac{\lambda}{M^2} \phi^6$

In [ ]:
# Función para armarse la matriz de monodromía y calcular los coeficientes de Floquet
def mu_phi6(k, q, y0, Tosc):
    phi0, phi_dot0 = y0

    def pert(t, y, k, q):
        phi, phi_dot, chi_k, chi_dot_k = y
        phi_ddot = - phi**5
        chi_ddot_k = - (k**2 + q * phi**2) * chi_k
        return [phi_dot, phi_ddot, chi_dot_k, chi_ddot_k]

    t_span = (0, Tosc)

    sol1 = solve_ivp(pert, t_span, [phi0, phi_dot0, 1.0, 0.0], t_eval=[Tosc], method='RK45', rtol=1e-6, args=(k, q))
    y_final1 = final_state(sol1)

    sol2 = solve_ivp(pert, t_span, [phi0, phi_dot0, 0.0, 1.0], t_eval=[Tosc], method='RK45', rtol=1e-6, args=(k, q))
    y_final2 = final_state(sol2)

    M = np.array([
        [y_final1[2], y_final2[2]],
        [y_final1[3], y_final2[3]]
    ])

    tr_M = np.trace(M)
    argument = np.abs(tr_M / 2.0)

    if argument > 1.0:
        mu = np.arccosh(argument) / Tosc
        return mu
    else:
        return 0.0
    
# Función para calcular el período de oscilación del inflatón
def oscilation_phi6(E):
    def integrand(phi):
        V = 1/6 * phi**6
        return 1.0 / np.sqrt(2 * (E - V))
    phimax = (6*E)**(1/6)
    integral, _ = quad(integrand, 0, phimax)
    Tosc = 4*integral
    return Tosc

In [ ]:
phi0 = 1
phi_dot0 = 0
y0 = (phi0, phi_dot0)
E = 0.5 * phi_dot0**2 + 1/6 * phi0**6

gs = np.linspace(0, 25, 200)
ks = np.logspace(-2, 0.5, 200)

G, K = np.meshgrid(gs, ks)
mu = np.zeros((len(ks), len(gs)), dtype=complex)
Tosc = oscilation_phi6(E)

for i, g in enumerate(tqdm(gs)):
    for j, k in enumerate(ks):
        mu[j, i] = mu_phi6(k, g, y0, Tosc)

In [ ]:
# Ahora grafiquemos el diagrama de inestabilidades
plt.figure(figsize=(10, 6))
plt.contourf(gs, ks, np.real(mu), levels=100, cmap='inferno')
plt.colorbar(label=r'$\mu_k$')
plt.xlabel(r'$q$')
plt.ylabel(r'$k$')
plt.title(r'Diagrama de inestabilidades de Floquet para $\frac{\lambda}{M^2} \phi^6$')
plt.tight_layout()

# Starobinsky

Tomamos un potencial de la forma:

$$\displaystyle \tilde{V} \left( \tilde{\phi} \right) = \frac{3}{4} \left[ 1 - \exp \left( - \sqrt{\frac{2}{3}} \tilde{\phi} \right) \right]^2$$

In [ ]:
# Función para armarse la matriz de monodromía y calcular los coeficientes de Floquet
def mu_starobinsky(k, q, y0, Tosc):
    phi0, phi_dot0 = y0

    def pert(t, y, k, q):
        phi, phi_dot, chi_k, chi_dot_k = y
        phi_ddot = - np.sqrt(3/2) * (np.exp(- np.sqrt(2/3) * phi) - np.exp(- 2*np.sqrt(2/3) * phi))
        chi_ddot_k = - (k**2 + q * phi**2) * chi_k
        return [phi_dot, phi_ddot, chi_dot_k, chi_ddot_k]

    t_span = (0, Tosc)

    sol1 = solve_ivp(pert, t_span, [phi0, phi_dot0, 1.0, 0.0], t_eval=[Tosc], method='RK45', rtol=1e-6, args=(k, q))
    y_final1 = final_state(sol1)

    sol2 = solve_ivp(pert, t_span, [phi0, phi_dot0, 0.0, 1.0], t_eval=[Tosc], method='RK45', rtol=1e-6, args=(k, q))
    y_final2 = final_state(sol2)

    M = np.array([
        [y_final1[2], y_final2[2]],
        [y_final1[3], y_final2[3]]
    ])

    tr_M = np.trace(M)
    argument = np.abs(tr_M / 2.0)

    if argument > 1.0:
        mu = np.arccosh(argument) / Tosc
        return mu
    else:
        return 0.0
    
# Función para calcular el período de oscilación del inflatón
def oscilation_starobinsky(E):
    def integrand(phi):
        V = 0.75 * (1 - np.exp(- np.sqrt(2/3) * phi))**2
        return 1.0 / np.sqrt(2 * (E - V))
    phimin = -1 * np.sqrt(3/2) * np.log(1 + np.sqrt(4*E/3))
    phimax = -1 * np.sqrt(3/2) * np.log(1 - np.sqrt(4*E/3))
    integral, _ = quad(integrand, phimin, phimax)
    Tosc = 2*integral
    return Tosc

In [ ]:
# Verifico si el archivo de inestabilidades existe, sino lo armo
filename = "mathieu_starobinsky_1.csv"
kappa = 1e-6

if not os.path.exists(filename):
    phi0 = np.sqrt(3/2) * np.log(1 + np.sqrt(2/3))
    phi_dot0 = - 3/4 * (np.sqrt(2/3)/(np.sqrt(2/3) + 1))
    y0 = (phi0, phi_dot0)
    E = 0.5 * phi_dot0**2 + 0.75 * (1 - np.exp(- np.sqrt(2/3) * phi0))**2

    qs = np.logspace(0, 4, 300)
    ks = np.logspace(-2, 1, 300)
    gs = []

    for q in qs:
        g = np.sqrt(q) * kappa
        gs.append(g)

    G, K = np.meshgrid(gs, ks)
    mu = np.zeros((len(ks), len(gs)), dtype=complex)
    Tosc = oscilation_starobinsky(E)

    for i, q in enumerate(tqdm(qs)):
        for j, k in enumerate(ks):
            mu[j, i] = mu_starobinsky(k, q, y0, Tosc)

    # Guardemos los resultados en un archivo CSV
    save_csv(gs, ks, mu, filename='mathieu_starobinsky_1')
    
else:
    data = np.loadtxt(filename, delimiter=",")
    gs = data[0, 1:]
    ks = data[1:, 0]
    mu = data[1:, 1:]

In [ ]:
# Ahora grafiquemos el diagrama de inestabilidades
plt.figure(figsize=(10, 6))
plt.contourf(gs, ks, np.real(mu), levels=100, cmap='inferno')
plt.colorbar(label=r'$\mu_k$')
plt.xlabel(r'$g$')
plt.ylabel(r'$k$')
plt.title(r'Diagrama de inestabilidades de Floquet para Starobinsky')
plt.yscale('log')
plt.xscale('log')
plt.tight_layout()
plt.savefig("Floquet_starobinsky_1.pdf", format = "pdf")